# Underwater Dataset – Visual Distortion Analysis

This notebook computes simple **visual distortion metrics** for all images in your underwater datasets (AquaCoop, OzFish, aquarium, deepfish, f4k, fish_416, fishclef, luderick, negatives). The goal is to quantify:

- turbidity / haze (via a dark-channel–based proxy)
- colour cast (red / green / blue channel ratios)
- contrast (RMS contrast of the grayscale image)
- blur (variance of the Laplacian)

The results are saved into a single CSV file that you can later join with your YOLO detection results to analyse which visual conditions harm performance.

## 1. Install and import dependencies

Run the cell below once to make sure the required Python packages are available.
If they're already installed in your environment, this will be fast.

In [1]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

## 2. Configuration

- `ROOT` is the **hard-coded path** to your `Audit` directory.
- `DATASET_DIRS` lists the subfolders that contain your individual datasets.

If your `Audit` folder lives somewhere else, edit `ROOT` accordingly before running the notebook.

In [2]:
candidates = [
    Path('/Users/Marco/Desktop/Audit'),
    Path.cwd(),
    Path.home() / 'Desktop' / 'Audit',
]
ROOT = next((p for p in candidates if p.exists()), candidates[0])
if not ROOT.exists():
    print(f\
)

DATASET_DIRS = [
    'AquaCoop',
    'OzFish',
    'aquarium',
    'deepfish',
    'f4k',
    'fish_416',
    'fishclef',
    'luderick',
    'negatives/deepfish_negatives',
]

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

ROOT

PosixPath('/Users/Marco/Desktop/Audit')

## 3. Define visual distortion metric functions

This section defines small helper functions to compute, for each image, a set of underwater-relevant distortion metrics. Together, these quantify the main visual degradations that influence cross-domain generalisation in underwater fish detection:

- `turbidity_score(img_bgr)`: a dark-channel–based proxy for haze/backscatter — higher means stronger turbidity.
- `color_cast(img_bgr)`: relative contribution of the B, G, and R channels, capturing wavelength-dependent color attenuation.
- `rms_contrast(img_gray)`: RMS contrast — lower values indicate flatter, low-visibility images.
- `blur_metric(img_gray)`: variance of Laplacian — lower values correspond to blurrier images with reduced edge detail.
- `compute_uiqm(img_bgr)`: the full UIQM score and its three components (`UICM`, `UISM`, `UIConM`), describing color fidelity, sharpness, and contrast in underwater conditions.
- `compute_uciqe(img_bgr)`: the UCIQE score, combining chroma dispersion, luminance contrast, and saturation to characterise underwater color degradation.

These metrics are widely adopted in underwater imaging research and allow us to quantitatively relate visual distortions to model performance across domains.


In [3]:
import numpy as np
import cv2



def turbidity_score(img_bgr: np.ndarray) -> float:
    """
    Simple turbidity / haze proxy using a dark channel prior.

    We use the mean value of the dark channel:
        higher dark-channel mean  -> more haze / turbidity

    So:
        higher turbidity_score -> more turbidity.
    """
    img = img_bgr.astype(np.float32) / 255.0

    min_channel = np.min(img, axis=2)

    kernel = np.ones((15, 15), np.uint8)
    dark_channel = cv2.erode(min_channel, kernel)

    return float(dark_channel.mean())


def color_cast(img_bgr: np.ndarray) -> dict:
    """
    Channel dominance: relative contribution of B, G, R.
    Underwater images often have low red, higher green/blue.

    Returns:
        dict with 'blue_ratio', 'green_ratio', 'red_ratio'
        that sum to ~1.0
    """
    img = img_bgr.astype(np.float32)
    b_mean = np.mean(img[:, :, 0])
    g_mean = np.mean(img[:, :, 1])
    r_mean = np.mean(img[:, :, 2])

    total = b_mean + g_mean + r_mean + 1e-6

    return {
        "blue_ratio": float(b_mean / total),
        "green_ratio": float(g_mean / total),
        "red_ratio": float(r_mean / total),
    }


def rms_contrast(img_gray: np.ndarray) -> float:
    """
    RMS contrast: standard deviation of a gray image, normalised to [0,1].

        higher -> higher global contrast
        lower  -> flatter / low-contrast image
    """
    return float(img_gray.astype(np.float32).std() / 255.0)


def blur_metric(img_gray: np.ndarray) -> float:
    """
    Focus/blur measure using variance of Laplacian.

        higher -> sharper image
        lower  -> blurrier image
    """
    lap = cv2.Laplacian(img_gray, cv2.CV_64F)
    return float(lap.var())



def compute_uicm(img_bgr: np.ndarray) -> float:
    """
    UICM: colourfulness / colour cast metric for underwater images.

    Based on red–green (RG) and yellow–blue (YB) differences.
    This is a simplified implementation following the spirit of
    Panetta et al. (2015).
    """
    img = img_bgr.astype(np.float32)
    B, G, R = img[:, :, 0], img[:, :, 1], img[:, :, 2]

    RG = R - G
    YB = 0.5 * (R + G) - B

    mu_RG, mu_YB = np.mean(RG), np.mean(YB)
    sigma_RG, sigma_YB = np.std(RG), np.std(YB)

    uicm = (-0.0268 * mu_RG) + (0.1586 * sigma_RG) \
           + (-0.0402 * mu_YB) + (0.1295 * sigma_YB)
    return float(uicm)


def compute_uism(img_bgr: np.ndarray) -> float:
    """
    UISM: sharpness / blur metric.

    Uses Sobel gradient magnitude averaged over channels.
        higher -> sharper (more edges / detail)
        lower  -> blurrier
    """
    img = img_bgr.astype(np.float32) / 255.0
    acc = 0.0
    for c in range(3):
        ch = img[:, :, c]
        gx = cv2.Sobel(ch, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(ch, cv2.CV_32F, 0, 1, ksize=3)
        g = np.sqrt(gx ** 2 + gy ** 2)
        acc += g.mean()
    return float(acc / 3.0)


def compute_uiconm(img_bgr: np.ndarray) -> float:
    """
    UIConM: contrast metric based on luminance statistics.

    This is a monotonic proxy based on mean and std of the gray image.
        higher -> higher contrast
        lower  -> flatter image
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    mu = gray.mean()
    sigma = gray.std()

    return float((mu ** 0.25) * (sigma ** 0.5))


def compute_uiqm(img_bgr: np.ndarray):
    """
    Full UIQM + its three components.

    Returns:
        uiqm_val, uicm, uism, uiconm
    """
    uicm = compute_uicm(img_bgr)
    uism = compute_uism(img_bgr)
    uiconm = compute_uiconm(img_bgr)

    uiqm_val = 0.0282 * uicm + 0.2953 * uism + 3.5753 * uiconm
    return float(uiqm_val), uicm, uism, uiconm



def compute_uciqe(img_bgr: np.ndarray) -> float:
    """
    UCIQE: Underwater Color Image Quality Evaluation.

    Combines:
        - sigma_c : chroma dispersion
        - con_l   : luminance contrast (percentile-based)
        - mu_s    : mean saturation

    Implementation based on Yang et al. (2015).
    Typical values are roughly in [0, ~5], depending on image content.
    """
    img = img_bgr.astype(np.float32) / 255.0

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L = lab[:, :, 0]
    a = lab[:, :, 1]
    b = lab[:, :, 2]

    a_c = a - 128.0
    b_c = b - 128.0

    chroma = np.sqrt(a_c ** 2 + b_c ** 2)
    sigma_c = chroma.std()

    L_flat = L.flatten()
    L_60 = np.percentile(L_flat, 60)
    L_40 = np.percentile(L_flat, 40)
    con_l = (L_60 - L_40) / 100.0

    eps = 1e-6
    sat = chroma / (L + eps)
    mu_s = sat.mean()

    uciqe_val = 0.4680 * sigma_c + 0.2745 * con_l + 0.2576 * mu_s
    return float(uciqe_val)


## 3.5 Define density and occlusion metrics

In addition to visual distortions, it is useful to describe how many fish are present in each image and how often they overlap. To do this, we derive two simple measures directly from the YOLO label files: the number of annotated fish per image (density) and a coarse occlusion score based on the average overlap between bounding boxes. These metrics help distinguish sparse scenes from crowded, heavily occluded ones, which is important for understanding cross-domain generalisation.


In [4]:
from pathlib import Path

def load_yolo_boxes(label_path: Path) -> np.ndarray:
    """
    Load YOLO labels from a .txt file and return an array of normalised
    [x_center, y_center, width, height] boxes. If the file does not exist
    or is empty, an empty array is returned.
    """
    if not label_path.exists():
        return np.zeros((0, 4), dtype=np.float32)

    try:
        data = np.loadtxt(str(label_path), ndmin=2)
    except Exception:
        return np.zeros((0, 4), dtype=np.float32)

    if data.size == 0 or data.shape[1] < 5:
        return np.zeros((0, 4), dtype=np.float32)

    boxes_xywhn = data[:, 1:5].astype(np.float32)
    return boxes_xywhn


def boxes_xywhn_to_xyxy(boxes_xywhn: np.ndarray) -> np.ndarray:
    """
    Convert normalised [x, y, w, h] to [x1, y1, x2, y2] in the same
    normalised coordinate space.
    """
    if boxes_xywhn.size == 0:
        return boxes_xywhn

    x, y, w, h = boxes_xywhn.T
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return np.stack([x1, y1, x2, y2], axis=1)


def mean_pairwise_iou(boxes_xywhn: np.ndarray) -> float:
    """
    Compute the mean pairwise IoU between all boxes in normalised space.
    If there are fewer than two boxes, the occlusion score is set to 0.0.
    """
    n = len(boxes_xywhn)
    if n < 2:
        return 0.0

    boxes = boxes_xywhn_to_xyxy(boxes_xywhn)
    ious = []
    for i in range(n):
        x1_i, y1_i, x2_i, y2_i = boxes[i]
        area_i = max(x2_i - x1_i, 0.0) * max(y2_i - y1_i, 0.0)
        if area_i <= 0:
            continue
        for j in range(i + 1, n):
            x1_j, y1_j, x2_j, y2_j = boxes[j]
            area_j = max(x2_j - x1_j, 0.0) * max(y2_j - y1_j, 0.0)
            if area_j <= 0:
                continue

            inter_x1 = max(x1_i, x1_j)
            inter_y1 = max(y1_i, y1_j)
            inter_x2 = min(x2_i, x2_j)
            inter_y2 = min(y2_i, y2_j)

            inter_w = max(inter_x2 - inter_x1, 0.0)
            inter_h = max(inter_y2 - inter_y1, 0.0)
            inter_area = inter_w * inter_h

            if inter_area <= 0:
                continue

            union_area = area_i + area_j - inter_area
            if union_area <= 0:
                continue

            iou = inter_area / union_area
            ious.append(iou)

    if not ious:
        return 0.0
    return float(np.mean(ious))


def compute_density_and_occlusion(img_path: Path) -> tuple[int, float]:
    """
    Compute:
      - density: number of annotated fish per image
      - occlusion: mean pairwise IoU between boxes (0 if fewer than two boxes)

    The function assumes YOLO .txt labels. If the image path lives under an
    'images' directory, the corresponding label is assumed to live under
    a parallel 'labels' directory with the same relative path. Otherwise,
    a .txt file next to the image is used.
    """
    label_path = img_path.with_suffix('.txt')

    parts = list(img_path.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        candidate = Path(*parts).with_suffix('.txt')
        if candidate.exists():
            label_path = candidate

    boxes_xywhn = load_yolo_boxes(label_path)
    density = int(len(boxes_xywhn))
    occlusion = mean_pairwise_iou(boxes_xywhn)
    return density, occlusion


## 4. Helper to collect all images in each dataset

We recursively walk each dataset directory under `ROOT` and collect all files with a supported image extension. This way we don't assume a particular subfolder structure beyond what you already have.

In [5]:
def get_all_images(dataset_root: Path):
    """Recursively collect all image files under a dataset root."""
    for p in dataset_root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            yield p

sample_root = ROOT / DATASET_DIRS[0]
print('Sample dataset root:', sample_root)
if sample_root.exists():
    sample_imgs = list(get_all_images(sample_root))[:5]
    print('First few images:', [str(p) for p in sample_imgs])
else:
    print('[WARN] Sample dataset root does not exist – check ROOT and DATASET_DIRS.')

Sample dataset root: /Users/Marco/Desktop/Audit/AquaCoop
First few images: ['/Users/Marco/Desktop/Audit/AquaCoop/images/AquaCoop_01217.jpg', '/Users/Marco/Desktop/Audit/AquaCoop/images/AquaCoop_00109.jpg', '/Users/Marco/Desktop/Audit/AquaCoop/images/AquaCoop_01203.jpg', '/Users/Marco/Desktop/Audit/AquaCoop/images/AquaCoop_00135.jpg', '/Users/Marco/Desktop/Audit/AquaCoop/images/AquaCoop_00653.jpg']


## 5. Compute distortion metrics for all datasets

This cell loops over every dataset listed in `DATASET_DIRS`, computes the metrics for each image, and stores them in a list of dictionaries. At the end, everything is assembled into a single `DataFrame`.

In [6]:
rows = []

for ds_rel in DATASET_DIRS:
    ds_root = ROOT / ds_rel
    if not ds_root.exists():
        print(f"[WARN] Dataset root not found: {ds_root}")
        continue

    dataset_name = ds_root.name
    img_paths = list(get_all_images(ds_root))
    print(f"Dataset '{dataset_name}': found {len(img_paths)} images")

    for img_path in tqdm(img_paths, desc=f"Processing {dataset_name}"):
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"[WARN] Could not read image: {img_path}")
            continue

        img_bgr = img
        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        t_score = turbidity_score(img_bgr)
        cc = color_cast(img_bgr)
        contrast = rms_contrast(img_gray)
        blur = blur_metric(img_gray)

        uiqm_val, uicm, uism, uiconm = compute_uiqm(img_bgr)
        uciqe_val = compute_uciqe(img_bgr)

        fish_count, mean_iou_overlap = compute_density_and_occlusion(img_path)

        rows.append({
            "dataset": dataset_name,
            "path_rel": str(img_path.relative_to(ROOT)),
            "turbidity": t_score,
            "blue_ratio": cc["blue_ratio"],
            "green_ratio": cc["green_ratio"],
            "red_ratio": cc["red_ratio"],
            "contrast": contrast,
            "blur_var_laplacian": blur,
            "uiqm": uiqm_val,
            "uicm": uicm,
            "uism": uism,
            "uiconm": uiconm,
            "uciqe": uciqe_val,
            "fish_count": fish_count,
            "mean_iou_overlap": mean_iou_overlap,
        })

df = pd.DataFrame(rows)
df.to_csv(ROOT / "distortions_all_datasets.csv", index=False)
df.head()


Dataset 'AquaCoop': found 1238 images


Processing AquaCoop: 100%|██████████| 1238/1238 [04:22<00:00,  4.71it/s]


Dataset 'OzFish': found 350 images


Processing OzFish: 100%|██████████| 350/350 [01:32<00:00,  3.80it/s]


Dataset 'aquarium': found 637 images


...

Dataset 'luderick': found 4276 images


Processing luderick: 100%|██████████| 4276/4276 [1:14:39<00:00,  1.05s/it]    


Dataset 'deepfish_negatives': found 2012 images


Processing deepfish_negatives:   0%|          | 0/2012 [00:00<?, ?it/s]/var/folders/3n/pg96hx_s297_yq_nk17lgfv40000gp/T/ipykernel_64446/567846090.py:13: UserWarning: loadtxt: input contained no data: "/Users/Marco/Desktop/Audit/negatives/deepfish_negatives/labels/deepfish_negatives_01280.txt"
  data = np.loadtxt(str(label_path), ndmin=2)
Processing deepfish_negatives:   0%|          | 1/2012 [00:04<2:29:27,  4.46s/it]/var/folders/3n/pg96hx_s297_yq_nk17lgfv40000gp/T/ipykernel_64446/567846090.py:13: UserWarning: loadtxt: input contained no data: "/Users/Marco/Desktop/Audit/negatives/deepfish_negatives/labels/deepfish_negatives_00820.txt"
  data = np.loadtxt(str(label_path), ndmin=2)
Processing deepfish_negatives:   0%|          | 2/2012 [00:08<2:20:20,  4.19s/it]/var/folders/3n/pg96hx_s297_yq_nk17lgfv40000gp/T/ipykernel_64446/567846090.py:13: UserWarning: loadtxt: input contained no data: "/Users/Marco/Desktop/Audit/negatives/deepfish_negatives/labels/deepfish_negatives_00834.txt"
  data

,dataset,path_rel,turbidity,blue_ratio,green_ratio,red_ratio,contrast,blur_var_laplacian,uiqm,uicm,uism,uiconm,uciqe,fish_count,mean_iou_overlap
0,AquaCoop,AquaCoop/images/AquaCoop_01217.jpg,0.115074,0.332942,0.384627,0.282431,0.066663,2.646765,0.622253,1.264695,0.016715,0.162687,4.712023,23,0.119487
1,AquaCoop,AquaCoop/images/AquaCoop_00109.jpg,0.052302,0.417080,0.323054,0.259867,0.031167,1.020925,0.372260,1.560304,0.008840,0.091083,11.818447,4,0.017392
2,AquaCoop,AquaCoop/images/AquaCoop_01203.jpg,0.130606,0.376721,0.326785,0.296494,0.054079,3.182700,0.597883,2.309086,0.019189,0.147428,5.063146,13,0.109645
3,AquaCoop,AquaCoop/images/AquaCoop_00135.jpg,0.027801,0.414098,0.311070,0.274832,0.021176,1.379734,0.257200,0.832843,0.009740,0.064565,138.845001,3,0.000000
4,AquaCoop,AquaCoop/images/AquaCoop_00653.jpg,0.051603,0.336251,0.382439,0.281310,0.047197,1.901370,0.457774,1.453581,0.013370,0.115469,14.477638,12,0.087872


## 6. Save metrics to CSV

Finally, we save the full table of distortion metrics to a CSV file in your `Audit` directory.
You can later join this CSV with your YOLO per-image evaluation results to analyse how performance varies with turbidity, colour cast, blur, etc.

In [11]:
notebook_dir = Path.cwd()
out_path_root = ROOT / 'distortions_all_datasets.csv'
out_path_local = notebook_dir / 'distortions_all_datasets.csv'
df.to_csv(out_path_root, index=False)
df.to_csv(out_path_local, index=False)
print(f"Saved CSV to: {out_path_root} and {out_path_local}")
out_path_root

Saved CSV to: /Users/Marco/Desktop/Audit/distortions_all_datasets.csv and /Users/Marco/Desktop/Audit/Ablation_study/distortions_all_datasets.csv


PosixPath('/Users/Marco/Desktop/Audit/distortions_all_datasets.csv')

## 7. Summarise distortions per dataset

In [16]:
import pandas as pd
df = pd.read_csv("/Users/Marco/Desktop/Audit/Ablation_study/distortions_all_datasets.csv")

summary = df.groupby("dataset").agg({
    "turbidity": ["mean", "std"],
    "contrast": ["mean", "std"],
    "blur_var_laplacian": ["mean", "std"],
    "blue_ratio": "mean",
    "green_ratio": "mean",
    "red_ratio": "mean",
    "uiqm": ["mean", "std"],
    "uciqe": ["mean", "std"],
    "uicm": ["mean", "std"],
    "uism": ["mean", "std"],
    "uiconm": ["mean", "std"],
    "fish_count": ["mean", "std"],
    "mean_iou_overlap": ["mean", "std"],
}).reset_index()

summary.to_csv(str(ROOT / "Ablation_study/dataset_distortion_summary_full.csv"), index=False)
summary

dataset turbidity            contrast            \
                           mean       std      mean       std   
0            AquaCoop  0.138182  0.100411  0.055266  0.020099   
1              OzFish  0.052927  0.078410  0.100450  0.035976   
2            aquarium  0.180264  0.095395  0.178778  0.046255   
3            deepfish  0.298371  0.120443  0.098178  0.036356   
4  deepfish_negatives  0.310566  0.120649  0.086846  0.030812   
5                 f4k  0.227783  0.094866  0.280905  0.039615   
6            fish_416  0.146478  0.098548  0.153885  0.056511   
7            fishclef  0.196665  0.069818  0.279280  0.041672   
8            luderick  0.135861  0.097948  0.136747  0.037579   

  blur_var_laplacian              blue_ratio green_ratio red_ratio  ...  \
                mean          std       mean        mean      mean  ...   
0           4.935778     5.053519   0.369395    0.337593  0.293012  ...   
1         249.041162   273.217338   0.473965    0.456368  0.069667  ...   
2         217.843774   325.633850   0.415615    0.323483  0.260902  ...   
3          53.564837    44.772868   0.378013    0.381734  0.240253  ...   
4          49.118174    38.658730   0.384060    0.375171  0.240769  ...   
5        2050.205872  1027.551390   0.331002    0.345227  0.323772  ...   
6         273.283729   420.247831   0.419801    0.356251  0.223948  ...   
7        1811.851456  1052.960293   0.332080    0.344991  0.322929  ...   
8         522.215123   103.733759   0.402521    0.454645  0.142834  ...   

        uicm                uism              uiconm           fish_count  \
        mean       std      mean       std      mean       std       mean   
0   1.869538  0.939046  0.021794  0.010471  0.146384  0.048367  11.060582   
1  13.755890  3.065266  0.115020  0.066795  0.250508  0.047952  21.542857   
2   7.718419  4.140612  0.137080  0.070465  0.319277  0.062031   7.568289   
3   8.809518  4.489049  0.073281  0.023900  0.257396  0.046197   3.432408   
4   8.277496  4.459765  0.068605  0.019055  0.242904  0.041154   0.000000   
5   9.929343  4.221033  0.387795  0.144532  0.446763  0.034116   3.846348   
6   8.446263  4.057918  0.163268  0.106305  0.297980  0.065996   2.326471   
7   9.857090  3.894416  0.372920  0.135736  0.436855  0.029715   1.585301   
8  16.541974  5.278859  0.107537  0.037229  0.305150  0.045214   2.205098   

             mean_iou_overlap            
         std             mean       std  
0   5.333082         0.073698  0.052914  
1  17.097634         0.076132  0.056537  
2   8.395143         0.047847  0.071716  
3   3.144170         0.018455  0.048745  
4   0.000000         0.000000  0.000000  
5   3.538521         0.008571  0.023976  
6   3.483926         0.019348  0.050056  
7   0.855049         0.003349  0.029070  
8   2.367575         0.009815  0.030335  

[9 rows x 24 columns]